# KWS eval part 1: Label posteriors with ZIPA
In this notebook I've given you several pieces of the puzzle for how to get KWS probability for Tira audio using the ZIPA model.
I've broken it down into 3 stages:
1. Loading KWS sentences and audio
2. Computing logits with ZIPA
3. Getting label posteriors for each sentence using CTC loss

We don't quite get to actual keyword hit probability in this exercise, but we'll get there next.

## Objective
As you go through this exercise, rather than just fill out the notebook, I'm asking you to create a python script (I've left a blank file in `scripts/eval/kws_ctc_eval.py`) that will get keyword probabilities for every sentence in the dataset.
Think about the components I've given you here, and how they can be arranged in a logical sequence so that you can write a single script that when run end-to-end will perform all the steps needed to evaluate the ZIPA model on our Tira KWS dataset.

## Next steps
After you finish this task the next objective will be to swap from naive label probability to keyword hit probability, and then to use the probabilities to compute evaluation metrics like mAP, F1 and AUROC.

In [1]:
from tira_kws.dataloading import load_capstone_kws_cuts, get_k2_dataloader
from tira_kws.models.zipa import load_zipa_small_crctc, decode_ctc, ctc_params
from tira_kws.models.zipa import load_zipa_large_transducer, decode_transducer, transducer_params
from tira_kws.constants import CAPSTONE_SUPERVISIONS, CAPSTONE_KEYWORDS, RECORD_LIST_CSV
from lhotse import SupervisionSet
import pandas as pd
import torch

/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1306: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, x, y, alpha):
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1315: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad):
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1512: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1551: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad: Tensor):


## Prior reading
Please read this [HuggingFace chapter](https://huggingface.co/learn/llm-course/en/chapter3/4) and [PyTorch tutorial](https://sebastianraschka.com/teaching/pytorch-1h/_) before trying to understand any of the code here, and use them as references if anything here is confusing.

## Keywords
Keywords are stored in a CSV file.
We can load this with Pandas.

In [2]:
df = pd.read_csv(CAPSTONE_KEYWORDS)
df.head()

,keyword,keyword_id,gloss
0,àpɾí,0.0,boy
1,ðɔ̀mɔ̀cɔ̀,1.0,man
2,lɛ́ŋgɛ́n,2.0,mother
3,ŋɔ̀ɽíŋgɔ́,3.0,donkey
4,mùðù,4.0,leopard


We'll save the list of keyword strings for easy reference later.

In [3]:
keywords = df['keyword'].tolist()
keywords

['àpɾí',
 'ðɔ̀mɔ̀cɔ̀',
 'lɛ́ŋgɛ́n',
 'ŋɔ̀ɽíŋgɔ́',
 'mùðù',
 'ùɾnɔ̀',
 'ðàŋàl',
 'kə̀və̀lɛ̀ðɔ́',
 'kàŋú',
 'íŋgá_nɔ̀nà']

## Sentence data
KWS sentence metadata are stored in JSONL (= Javascript Object Notation Lines), where every line is a single JSON object which is written similarly to a Python dictionary and can be read as a Python dictionary directly. Let's print the first two lines of the JSONL file using the `json` library for formatting.

In [4]:
import json
with open(CAPSTONE_SUPERVISIONS) as f:
    lines = f.readlines()[:1]
json_objects = [json.loads(line) for line in lines]
json_objects_formatted = [
    json.dumps(obj, indent=2, ensure_ascii=False)
    for obj in json_objects
]
jsonl_header = "\n".join(json_objects_formatted)
print(jsonl_header)

{
  "id": "àpɾí_484_positive",
  "start": 1969.12,
  "duration": 2.851,
  "channel": 0,
  "supervisions": [
    {
      "id": "àpɾí_484_positive",
      "recording_id": "HH02122021",
      "start": 0.0,
      "duration": 2.851,
      "channel": 0,
      "text": "àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀",
      "language": "tic",
      "speaker": "Himidan",
      "custom": {
        "fst_text": "àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀",
        "gloss": "boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]",
        "root": "àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀",
        "translation": "The boy will pull me here tomorrow.",
        "keyword": "àpɾí",
        "norm_text": "àpɾí jáŋɛ̂_və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀",
        "record_type": "positive"
      }
    }
  ],
  "recording": {
    "id": "HH02122021",
    "sources": [
      {
        "type": "file",
        "channels": [
          0
        ],
        "source": "/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav"
      

Rather than loading JSONL metadata manually, we use the Lhotse library which is designed for managing speech datasets.
The function `tira_kws/dataloading.py#load_capstone_kws_cuts()` uses Lhotse to load KWS data into a `CutSet` object (i.e. a set of audio snippets cut out from the source audio).
As we can see, the cuts contain the same data as the JSONL file above.

Read the [Lhotse documentation](https://lhotse.readthedocs.io/en/latest/cuts.html) for more information on `Cut` and `CutSet` objects.
By default Lhotse returns a generator rather than a list.
To load cuts as a list instead (so we can index them), call `cuts.to_eager()`.

In [5]:
cuts = load_capstone_kws_cuts()
cuts = cuts.trim_to_supervisions()
cuts = cuts.to_eager()
cuts[0]

MonoCut(id='àpɾí_484_positive', start=1969.12, duration=2.851, channel=0, supervisions=[SupervisionSegment(id='àpɾí_484_positive', recording_id='HH02122021', start=0.0, duration=2.851, channel=0, text='àpɾí jǎŋîvlɛ̀ðɔ́ ǹdòbàgɛ̀', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG] aux pull[CLJ,VENT,IPFV,1SG.OBJ,aux] tomorrow[]', 'root': 'àpɾí ŋgá vəlɛð nd̪ɔ̀bàgɛ̀', 'translation': 'The boy will pull me here tomorrow.', 'keyword': 'àpɾí', 'norm_text': 'àpɾí jáŋɛ̂_və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH02122021', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH02122021.wav')], sampling_rate=16000, num_samples=55650022, duration=3478.126375, channel_ids=[0], transforms=None), custom={'fst_text': 'àpɾí jáŋɛ̂ və̀lɛ̀ðɔ́ nd̪ɔ̀bàgɛ̀', 'gloss': 'boy[NOM,SG]

## Data filtering
It's useful to load all sentences for a particular keyword rather than all sentences at once.

In [6]:
keyword = keywords[1]
keyword_cuts = cuts.filter(
    lambda cut: cut.custom.get('keyword', None) == keyword
)
keyword_cuts = keyword_cuts.to_eager()
keyword, keyword_cuts

('ðɔ̀mɔ̀cɔ̀', CutSet(len=20) [underlying data type: <class 'list'>])

We can also load only the negative records from the dataset.

In [7]:
negative_cuts = cuts.filter(
    lambda cut: cut.custom.get('record_type', None) == 'negative'
)
negative_cuts = negative_cuts.to_eager()
negative_cuts

CutSet(len=100) [underlying data type: <class 'list'>]

## Dataloading and batching
We use a dataloader to serve batches to the model.

In [8]:
dataloader = get_k2_dataloader(cuts)
dataloader

The dataloader batches the data into a roughly fixed size (no more than 32 records by default).
Each batch has 'inputs' (the audio features to be fed to the model, stored as torch tensors) and 'supervisions' (the metadata for each record in the batch).

See [Torch documentation](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) for more on dataloaders.

In [9]:
for i, batch in enumerate(dataloader):
    if i >= 5:
        break
    print(batch.keys())
    # input shape is batch_size, sequence_length, feature_dimension
    print(batch['inputs'].shape)

dict_keys(['inputs', 'supervisions'])
torch.Size([32, 231, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 186, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 155, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([32, 209, 80])
dict_keys(['inputs', 'supervisions'])
torch.Size([10, 727, 80])


Let's look more closely at the metadata stored under "supervisions."

In [10]:
first_batch = next(iter(dataloader))
supervisions = first_batch['supervisions']
supervisions.keys()

dict_keys(['text', 'sequence_idx', 'start_frame', 'num_frames', 'cut'])

All records in a batch are automatically padded to the same length, but they were not the same length originally.
Lhotse includes a 'num_frames' field that lets us know the original length of each record.

In [11]:
supervisions['num_frames']

tensor([231, 230, 228, 227, 227, 226, 225, 224, 224, 224, 224, 223, 223, 221,
        221, 219, 218, 218, 218, 217, 216, 215, 215, 215, 215, 214, 213, 212,
        211, 211, 210, 210], dtype=torch.int32)

`batch['supervisions']['cut']` is a list of `lhotse.MonoCut` objects, one for each record in the batch.
We can access the record's metadata (transcription, keyword, whether it's positive or negative) from the `.custom` field of the `MonoCut` object.

In [12]:
print(len(supervisions['cut']))
print(supervisions['cut'][0])

32
MonoCut(id='kàŋú_11178_positive', start=1404.27, duration=2.31, channel=0, supervisions=[SupervisionSegment(id='kàŋú_11178_positive', recording_id='HH20220919', start=0.0, duration=2.31, channel=0, text='kúkù kàŋú t̪ólìɲà vɽàn', language='tic', speaker='Himidan', gender=None, custom={'fst_text': 'kúkù kàŋú t̪ólèɲá ávàn', 'gloss': '1st.born.male.child[NOM,SG] see[CLG,VENT,PFV] lion[ACC,SG,left_h] very.much[left_h]', 'root': 'kúkù aŋ t̪òlé àvàn', 'translation': 'even Kuku saw the lion OR Kuku even/also saw the lion', 'keyword': 'kàŋú', 'norm_text': 'kúkù kàŋú t̪ólèɲá vɽàn', 'record_type': 'positive'}, alignment=None)], features=None, recording=Recording(id='HH20220919', sources=[AudioSource(type='file', channels=[0], source='/mnt/LocalStorage/mjsimmons/datasets/tira_elicitation/audio/HH20220919.wav')], sampling_rate=16000, num_samples=38687938, duration=2417.996125, channel_ids=[0], transforms=None), custom={'fst_text': 'kúkù kàŋú t̪ólèɲá á

In [13]:
print(supervisions['cut'][0].custom)

{'fst_text': 'kúkù kàŋú t̪ólèɲá ávàn', 'gloss': '1st.born.male.child[NOM,SG] see[CLG,VENT,PFV] lion[ACC,SG,left_h] very.much[left_h]', 'root': 'kúkù aŋ t̪òlé àvàn', 'translation': 'even Kuku saw the lion OR Kuku even/also saw the lion', 'keyword': 'kàŋú', 'norm_text': 'kúkù kàŋú t̪ólèɲá vɽàn', 'record_type': 'positive', 'dataloading_info': {'rank': 0, 'world_size': 1, 'worker_id': None}}


## Model
I've sequestered some of the code from ZIPA in `tira_kws/models/zipa.py` which we can use to load some of the ZIPA models.
For now let's stick with Zipa-Cr-Small.
Don't forget to put the model to GPU after loading, otherwise inference will be extremely slow.

In [14]:
model = load_zipa_small_crctc()
# model = load_zipa_large_transducer()
model = model.to('cuda')
type(model)

model.AsrModel

Here's a sample code snippet for getting CTC logits for a batch from the dataloader.

In [15]:
first_batch = next(iter(dataloader))

with torch.no_grad(): # what does 'no_grad' mean and why are we using it here?
    inputs = first_batch['inputs'].to('cuda')
    input_lengths = first_batch['supervisions']['num_frames'].to('cuda')
    # why put inputs and input_lens to cuda?
    # Hint: try running the snippet without `.to('cuda')`

    # Embeddings = hidden representation prior to output layer
    embeds, output_lengths = model.forward_encoder(inputs, input_lengths)
    # CTC output = logits from final layer
    ctc_logits = model.ctc_output(embeds)
    

## Tokenization
Before we can get keyword probabilities, we need to encode our keywords as token ids.
We can do this using the ZIPA sentencepiece tokenizer.

In [16]:
import sentencepiece
from tira_kws.constants import ZIPA_SENTENCEPIECE_MODEL

tokenizer = sentencepiece.SentencePieceProcessor()
tokenizer.load(str(ZIPA_SENTENCEPIECE_MODEL))

True

In [17]:
tokenizer.encode(['àpɾí', 'jícə̀lò'])

[[3, 4, 2, 19, 80, 12, 2], [3, 13, 12, 2, 6, 50, 2, 15, 18, 2]]

In [18]:
# TODO: encode every keyword from the first batch
# YOUR CODE HERE
encoded_batch_labels = ...

## Prebaked decoding functions
Let's try inference using the `decode_one_batch` function (masked as `decode_ctc`)

In [19]:
with torch.no_grad():
    outputs = decode_ctc(params=ctc_params, model=model, batch=first_batch, sp=tokenizer)
    # outputs = decode_transducer(params=transducer_params, model=model, batch=first_batch, sp=tokenizer)
outputs

{'ctc-greedy-search': [['ɒʎɒʎɒcɒʎɒʜɒʎcʎcɒcʎcɒʎɒʎɒʎɒʎɒʎɒpɒʎɒʎɛɒʎɛɒʎɒɛʎɛɒɛɒɛɒcɒʎɒ'],
  ['cʎcʎɒcʎɒʎɒʎɒʎɛʎʜʎɛʎpⱱɛⱱpɛʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒβɒɛʎɒ'],
  ['ɒʎɒʎɒɛʎɒcɒʎɒcɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒ'],
  ['ɒʎɒʎɒʎɒʎpʎɒcɒʎɒ', 'cɒⱱ', 'ɒpɒcʎcʎɒʎcɒʎɒʎɒʎɒʎcɒcʎɒʎɛʎcʎɒcʎɒʎɒ'],
  ['ɒʎcⱱcɒcⱱpⱱpⱱcpcpʎcʎcʎcʎcɒcpɒpcʟcɒⱱcⱱcɒcⱱɒʜcɒcɒ'],
  ['ʎɛɒɛɒʎɒʎɒʎɒʎɒʎɒʎɒɛɒʎɒʎɒʎɛʎɒʎɛʎʜʎɒʜɒʎɛʐɒʎkɒʎɒɛɒʎɒɛʎɒʎɛɒʎɒ'],
  ['ɒʎɒcɒcɒʎɒʎɒʎɒɛʜʎɒʎʜʎɒʜɒʎɒʜɒʎɒʎɒʜʎɒ'],
  ['ɒʎɒʎɒʎcɒcʎɒʎpɛɒcɒʎɒcɒʎɒcɛcɒɛɒɛɒʎɛɒʎɛʎɒʎɛɒʎɒcʎɒʎɒcɒcɒcɒʎɒʎɒʎɒʎɒʎʜ'],
  ['ɒʎɒʎcɒcɒcɒcɒcɒcɒcɒcʎɒcɒʎɒcʎɒcɒcɒʎcʎcɒcɒcʎɒʎɒ'],
  ['ɒʎɒʎɒʎɒʎɒʐʎɒʎʜʎɒʎcʎɒʐʎɒʎʐʎcpʎpɛpʎpʎʐɛʎɒɛʎɛʎɒʎcɒ'],
  ['cʎcʎɒʎʐʕɛɒɛɒpɒʎɛʎɛʎɛʎɛʎɒʎɒʎɒʎɒʎ̚ʐʎɛʎɛʎɛʎɛʎɒcʎɒʎɒʎɛʎɛʎɛʎɒʎɒʎɒ'],
  ['ɒʎɒʎɒʎɒʎɒʎɒʎɒʎɨɒcɒcɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎɒʎ'],
  ['ɒʎɒʎɒ', 'ɒ', 'ɒɛɒɛʎɒpɒpʎpcɒ', 'ɒ', 'ɒʎɒʎɒʎɒ'],
  ['ɒʎɒʎɒʎɒʎɒʎɒcɒʎɒʎcʎɒʎɒʎɒcɒʎɒʎɒʎʜʎɒʎɒʎɒʎɒ'],
  ['ɒʎɒʎɒcɒcɒ', 'ɒ', 'ɒʎɒʎɒ', 'ɒpɒɛʎɒʎɒ', 'ʎɒpɒʎɒʎɒʎɒʎɒʎɒʎɒʎcʎɒʎɒʎɒ'],
  ['ɒʎɒʎɒʎɒʎɒcɒʎɒʎɒʎɛʎʐʎɒɘɒʎɒcɒʎɛɒʎɒʎɒʎɒʎɒ'],
  ['ʎɒɛʎcʎcɛʎɒʎɒɛɒʎɒʎʜʎɛʎɒɛɨʎɛɒʎɒʎɒcʎcʎɒcɒʎɒʎɒʎɒɛɒɛʎɒʎɒcʎɒʎɒ'],
  ['ʎɒʎɒʎɛʎɛʐɛ

## CTC loss
CTC loss is equivalent to the probability of a label given the logits.
For KWS inference we want to get the probability of a label occurring *within* audio (along other speech), so this isn't quite what we're looking for but it's a good first step.
We'll this approach working before we think about how to get the KWS probability.

To get CTC loss, we need the label posteriors, targets(=labels), input lengths and target lengths.
To convert logits to probabilities, use softmax.(https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html#torch.nn.functional.log_softmax).


See the [PyTorch documentation on CTC loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.ctc_loss.html#torch.nn.functional.ctc_loss) for more info.

In [20]:
output_lengths.shape, ctc_logits.shape

(torch.Size([32]), torch.Size([32, 112, 127]))

In [53]:
from torch.nn.functional import log_softmax
from torch.nn import CTCLoss

ctc_loss = CTCLoss(reduction='none')

batch_size = ctc_logits.shape[0]

# replace with the actual encoded keywords from the batch computed above
encoded_keyword = tokenizer.encode('kúkùŋú')
targets = torch.tensor([encoded_keyword]*batch_size)
target_lengths = torch.tensor([len(encoded_keyword)]*batch_size)

# TODO: cast logits into probabilities
ctc_probs = log_softmax(ctc_logits, dim=2) # your code here

loss = ctc_loss(
    # permute from batch_size, length, vocab_size -> length, batch_size, vocab_size
    # to match the input shape expected by CTC loss
    ctc_probs.permute(1, 0, 2),
    targets,
    output_lengths,
    target_lengths,
)
loss, loss.shape

(tensor([452.2979, 455.2695, 464.4803, 446.3582, 452.5854, 461.0683, 430.7468,
         437.1606, 463.3562, 474.4314, 449.4125, 450.7033, 388.9162, 435.0609,
         398.8263, 419.6163, 430.1292, 436.3737, 373.8921, 392.6234, 407.8280,
         395.5663, 415.6659, 359.3753, 420.9825, 443.2006, 402.2831, 415.7921,
         415.3244, 388.8591, 373.9478, 403.6741], device='cuda:0'),
 torch.Size([32]))

## Label probability vs keyword probability
So now we've got the loss, which is equivalent to the probability of the label given the audio.
But this isn't enough for KWS.
We're not computing the probability that the audio *is* the keyword but that the audio *contains* the keyword.
To do this, we change the label so that any speech **OR** silence can be matched before or after the keyword.
This is the keyword/background model from last Fall.

How does CTC loss relate to keyword probability, and what is missing to convert it to KWS probability?
Hint: CTC loss is calculated by the intersection of two WFST graphs.
What are those graphs and what do they represent?
When doing KWS, how do the graphs differ?
I'd suggest drawing the graphs, and their resulting intersection, out using pen and paper.

Refer to [Xi et al 2025](10.1109/ICASSP49660.2025.10889332) for more explanation of KWS graph decoding.

## Lexicon-constrained decoding with Torchaudio
Define lexicon and token files and pass as decoding args.

In [26]:
from torchaudio.models.decoder import ctc_decoder

tokens_file = 'tokens.tmp'
keywords_file = 'keywords.tmp'

In [51]:
tokens = [tokenizer.decode(i) for i in range(127)]
tokens[0] = '-'
tokens[3] = '|'
with open(tokens_file, 'w') as f:
    f.write("\n".join(tokens))
with open(tokens_file) as f:
    print(f.read()[:50])

-
<sos/eos>
 ⁇ 
|
a
b
c
d
e
f
g
h
i
j
k
l
m
n
o
p



In [46]:
with open(keywords_file, 'w') as f:
    for word in keywords:
        f.write(word)
        f.write("\t")
        word_tokens = [c for c in word if c in tokens]
        f.write(" ".join(word_tokens))
        f.write("\n")
with open(keywords_file) as f:
    print(f.read())

àpɾí	a p ɾ i
ðɔ̀mɔ̀cɔ̀	ð ɔ m ɔ c ɔ
lɛ́ŋgɛ́n	l ɛ ŋ g ɛ n
ŋɔ̀ɽíŋgɔ́	ŋ ɔ ɽ i ŋ g ɔ
mùðù	m u ð u
ùɾnɔ̀	u ɾ n ɔ
ðàŋàl	ð a ŋ a l
kə̀və̀lɛ̀ðɔ́	k ə v ə l ɛ ð ɔ
kàŋú	k a ŋ u
íŋgá_nɔ̀nà	i ŋ g a n ɔ n a



In [52]:
decoder = ctc_decoder(
    lexicon=keywords_file,
    tokens=tokens_file,
    blank_token=tokens[0],
    sil_token=tokens[3],
    lm=None,
    lm_dict=None,
)

In [55]:
emissions = ctc_probs.cpu()

decoder(emissions)

[[CTCHypothesis(tokens=tensor([ 3, 15, 51, 35, 10, 51, 17,  3]), words=['lɛ́ŋgɛ́n'], score=-371.57917165756226, timesteps=tensor([  0,   4,   5,  39,  40,  41, 112, 113], dtype=torch.int32))],
 [CTCHypothesis(tokens=tensor([ 3, 15, 51, 35, 10, 51, 17,  3]), words=['lɛ́ŋgɛ́n'], score=-370.56555938720703, timesteps=tensor([  0,   1,   2,  53,  54,  55, 110, 111], dtype=torch.int32))],
 [CTCHypothesis(tokens=tensor([ 3, 15, 51, 35, 10, 51, 17,  3]), words=['lɛ́ŋgɛ́n'], score=-391.19622015953064, timesteps=tensor([ 0,  1,  2, 34, 36, 37, 82, 83], dtype=torch.int32))],
 [CTCHypothesis(tokens=tensor([ 3, 15, 51, 35, 10, 51, 17,  4, 19, 80, 12,  3, 15, 51, 35, 10, 51, 17,
           3]), words=['lɛ́ŋgɛ́n', 'àpɾí', 'lɛ́ŋgɛ́n'], score=-388.0519959926605, timesteps=tensor([  0,   1,   2,   9,  10,  11,  17,  18,  19,  33,  34,  35,  54,  55,
          108, 109, 110, 112, 113], dtype=torch.int32))],
 [CTCHypothesis(tokens=tensor([ 3, 15, 51, 35, 10, 51, 17,  3, 15, 51, 35, 10, 51, 17,  3]), wor